# MIT 805 Big Data Semester Project
## Part 1 - EDA in SQL (DuckDB)

This notebook mirrors the pandas EDA in `MIT805_Part1_HVFHV_EDA.ipynb` using SQL over the NYC TLC **HVFHV** Parquet files, queried directly from the official TLC server with DuckDB + JupySQL.

### Approach
1. **Create a view** over the remote Parquet file with derived fields (company, duration, wait time, hour, day). A view stores no data — DuckDB pushes column/filter selections down to the Parquet file, so each query fetches only what it needs.
2. **Profile** the data: schema, row counts, descriptive statistics, missing values, anomalies.
3. **Aggregate** for patterns: demand by hour/day, company shares, distance–fare correlation.

### Data Source
NYC TLC — official page: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  
Direct files: `https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_YYYY-MM.parquet`

## 1. Setup: DuckDB connection and JupySQL

In [3]:
import duckdb

# Load the SQL extension
%load_ext sql

# Create an in-memory DuckDB connection
conn = duckdb.connect()

# httpfs enables reading Parquet files over HTTPS
conn.execute("INSTALL httpfs; LOAD httpfs;")

# Same representative month as the pandas EDA notebook (Section 2.6)
EDA_SAMPLE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-06.parquet"
print("Querying:", EDA_SAMPLE_URL)

# Pass the connection to JupySQL
%sql conn --alias duckdb

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
Querying: https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-06.parquet


In [4]:
%%sql
SELECT 'Hello DuckDB!' AS greeting;

Running query in 'duckdb'

greeting
Hello DuckDB!


## 2. Schema inspection

`DESCRIBE` reads only the Parquet footer metadata, so this is fast even on a multi-GB remote file.

In [5]:
%%sql
DESCRIBE SELECT * FROM read_parquet('{{EDA_SAMPLE_URL}}');

Running query in 'duckdb'

column_name,column_type,null,key,default,extra
hvfhs_license_num,VARCHAR,YES,None,None,None
dispatching_base_num,VARCHAR,YES,None,None,None
originating_base_num,VARCHAR,YES,None,None,None
request_datetime,TIMESTAMP,YES,None,None,None
on_scene_datetime,TIMESTAMP,YES,None,None,None
pickup_datetime,TIMESTAMP,YES,None,None,None
dropoff_datetime,TIMESTAMP,YES,None,None,None
PULocationID,INTEGER,YES,None,None,None
DOLocationID,INTEGER,YES,None,None,None
trip_miles,DOUBLE,YES,None,None,None


## 3. Reusable view with derived fields

The view adds the same derived fields as Section 4 of the pandas notebook: readable company label, trip duration, rider wait time and calendar features.

In [6]:
%%sql
CREATE OR REPLACE VIEW trips AS
SELECT
    *,
    CASE hvfhs_license_num
        WHEN 'HV0002' THEN 'Juno'
        WHEN 'HV0003' THEN 'Uber'
        WHEN 'HV0004' THEN 'Via'
        WHEN 'HV0005' THEN 'Lyft'
        ELSE hvfhs_license_num
    END AS company,
    date_diff('second', pickup_datetime, dropoff_datetime) / 60.0 AS trip_duration_minutes,
    date_diff('second', request_datetime, on_scene_datetime) / 60.0 AS wait_time_minutes,
    EXTRACT(hour FROM pickup_datetime) AS pickup_hour,
    dayname(pickup_datetime) AS pickup_day
FROM read_parquet('{{EDA_SAMPLE_URL}}');

Running query in 'duckdb'

Count


## 4. Row count and descriptive statistics

In [7]:
%%sql
SELECT
    COUNT(*) AS records,
    ROUND(AVG(trip_miles), 2)               AS mean_miles,
    ROUND(MEDIAN(trip_miles), 2)            AS median_miles,
    ROUND(MAX(trip_miles), 2)               AS max_miles,
    ROUND(AVG(trip_duration_minutes), 2)    AS mean_duration_min,
    ROUND(MEDIAN(trip_duration_minutes), 2) AS median_duration_min,
    ROUND(AVG(base_passenger_fare), 2)      AS mean_fare,
    ROUND(MEDIAN(base_passenger_fare), 2)   AS median_fare,
    ROUND(MIN(base_passenger_fare), 2)      AS min_fare,
    ROUND(MAX(base_passenger_fare), 2)      AS max_fare
FROM trips;

Running query in 'duckdb'

records,mean_miles,median_miles,max_miles,mean_duration_min,median_duration_min,mean_fare,median_fare,min_fare,max_fare
19868009,5.13,3.04,321.55,20.34,16.33,28.03,20.01,-58.43,1450.51


## 5. Data quality

### 5.1 Missing `on_scene_datetime` by company

`COUNT(col)` skips NULLs, so `COUNT(*) - COUNT(col)` counts missing values. Missingness is structural: some bases do not report the on-scene time.

In [8]:
%%sql
SELECT
    company,
    COUNT(*) AS trips,
    COUNT(*) - COUNT(on_scene_datetime) AS missing_on_scene,
    ROUND(100.0 * (COUNT(*) - COUNT(on_scene_datetime)) / COUNT(*), 2) AS missing_pct
FROM trips
GROUP BY company
ORDER BY missing_pct DESC;

Running query in 'duckdb'

company,trips,missing_on_scene,missing_pct
Lyft,5700648,0,0.0
Uber,14167361,0,0.0


### 5.2 Anomaly counts

`COUNT(*) FILTER (condition)` counts each anomaly type in a single scan.

In [9]:
%%sql
SELECT
    COUNT(*) FILTER (trip_duration_minutes <= 0)   AS non_positive_duration,
    COUNT(*) FILTER (trip_duration_minutes > 1440) AS duration_over_24h,
    COUNT(*) FILTER (trip_miles = 0)               AS zero_distance,
    COUNT(*) FILTER (trip_miles > 100)             AS distance_over_100mi,
    COUNT(*) FILTER (base_passenger_fare < 0)      AS negative_fare,
    COUNT(*) FILTER (base_passenger_fare > 500)    AS fare_over_500,
    COUNT(*) FILTER (wait_time_minutes < 0)        AS negative_wait
FROM trips;

Running query in 'duckdb'

non_positive_duration,duration_over_24h,zero_distance,distance_over_100mi,negative_fare,fare_over_500,negative_wait
0,0,2500,2216,487,591,302633


## 6. Cleaned view

Same plausibility filters as Section 6.3 of the pandas notebook: duration in (0, 1440] minutes, distance in (0, 100] miles, base fare in [0, 500] dollars.

In [10]:
%%sql
CREATE OR REPLACE VIEW trips_clean AS
SELECT *
FROM trips
WHERE trip_duration_minutes > 0 AND trip_duration_minutes <= 1440
  AND trip_miles > 0 AND trip_miles <= 100
  AND base_passenger_fare >= 0 AND base_passenger_fare <= 500;

Running query in 'duckdb'

Count


In [11]:
%%sql
-- Before/after cleaning summary
SELECT
    (SELECT COUNT(*) FROM trips)       AS raw_records,
    (SELECT COUNT(*) FROM trips_clean) AS clean_records,
    ROUND(100.0 * ((SELECT COUNT(*) FROM trips) - (SELECT COUNT(*) FROM trips_clean))
          / (SELECT COUNT(*) FROM trips), 2) AS excluded_pct;

Running query in 'duckdb'

raw_records,clean_records,excluded_pct
19868009,19862685,0.03


## 7. Patterns

### 7.1 Distance–fare correlation, raw vs clean

In [12]:
%%sql
SELECT
    'raw' AS dataset,
    COUNT(*) AS records,
    ROUND(corr(trip_miles, base_passenger_fare), 4) AS distance_fare_corr
FROM trips
UNION ALL
SELECT
    'clean',
    COUNT(*),
    ROUND(corr(trip_miles, base_passenger_fare), 4)
FROM trips_clean;

Running query in 'duckdb'

dataset,records,distance_fare_corr
raw,19868009,0.8524
clean,19862685,0.8481


### 7.2 Demand and trip characteristics by pickup hour

In [13]:
%%sql
SELECT
    pickup_hour,
    COUNT(*) AS trips,
    ROUND(AVG(trip_miles), 2) AS avg_miles,
    ROUND(AVG(base_passenger_fare), 2) AS avg_fare,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_min
FROM trips_clean
GROUP BY pickup_hour
ORDER BY pickup_hour;

Running query in 'duckdb'

pickup_hour,trips,avg_miles,avg_fare,avg_duration_min
0,750851,5.55,27.62,18.58
1,533071,5.44,25.9,17.45
2,387843,5.52,25.87,17.04
3,305322,6.1,27.65,17.36
4,314013,7.07,32.78,18.15
5,369198,7.21,31.25,18.52
6,540895,6.35,28.7,18.98
7,792107,5.27,27.72,19.46
8,987225,4.65,26.92,19.26
9,963248,4.69,25.91,19.32


### 7.3 Trips by day of week

In [14]:
%%sql
SELECT
    pickup_day,
    COUNT(*) AS trips,
    ROUND(AVG(trip_miles), 2) AS avg_miles,
    ROUND(AVG(base_passenger_fare), 2) AS avg_fare
FROM trips_clean
GROUP BY pickup_day
ORDER BY CASE pickup_day
    WHEN 'Monday' THEN 1 WHEN 'Tuesday' THEN 2 WHEN 'Wednesday' THEN 3
    WHEN 'Thursday' THEN 4 WHEN 'Friday' THEN 5 WHEN 'Saturday' THEN 6
    WHEN 'Sunday' THEN 7 END;

Running query in 'duckdb'

pickup_day,trips,avg_miles,avg_fare
Monday,2916157,5.13,27.33
Tuesday,2492011,4.9,28.21
Wednesday,2634132,5.01,29.46
Thursday,2699368,5.12,29.92
Friday,2817746,5.08,28.32
Saturday,3029978,5.04,26.27
Sunday,3273293,5.45,26.91


### 7.4 Company breakdown with market share

In [15]:
%%sql
SELECT
    company,
    COUNT(*) AS trips,
    ROUND(AVG(trip_miles), 2) AS avg_miles,
    ROUND(AVG(base_passenger_fare), 2) AS avg_fare,
    ROUND(AVG(driver_pay), 2) AS avg_driver_pay,
    ROUND(AVG(tips), 2) AS avg_tips,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS market_share_pct
FROM trips_clean
GROUP BY company
ORDER BY trips DESC;

Running query in 'duckdb'

company,trips,avg_miles,avg_fare,avg_driver_pay,avg_tips,market_share_pct
Uber,14162916,5.25,29.02,22.27,1.22,71.3
Lyft,5699769,4.77,25.4,19.94,1.27,28.7


### 7.5 Rider wait time by hour (request → on-scene)

Bounded to plausible waits (0–60 min); rows with missing `on_scene_datetime` are excluded automatically because the derived wait time is NULL.

In [16]:
%%sql
SELECT
    pickup_hour,
    COUNT(*) AS trips_with_wait,
    ROUND(AVG(wait_time_minutes), 2) AS avg_wait_min
FROM trips_clean
WHERE wait_time_minutes >= 0 AND wait_time_minutes <= 60
GROUP BY pickup_hour
ORDER BY pickup_hour;

Running query in 'duckdb'

pickup_hour,trips_with_wait,avg_wait_min
0,749691,4.97
1,532156,4.55
2,386161,4.75
3,296763,5.06
4,290796,5.68
5,344313,4.4
6,517682,4.04
7,764824,4.44
8,961083,4.48
9,940249,3.89


### 7.6 Spatial view: top pickup zones

The TLC taxi-zone lookup CSV is joined directly in SQL — `read_csv_auto` works over HTTPS just like `read_parquet`.

In [17]:
%%sql
SELECT
    z.Borough AS borough,
    z.Zone AS zone,
    COUNT(*) AS pickups
FROM trips_clean t
JOIN read_csv_auto('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv') z
  ON t.PULocationID = z.LocationID
GROUP BY z.Borough, z.Zone
ORDER BY pickups DESC
LIMIT 15;

Running query in 'duckdb'

borough,zone,pickups
Queens,LaGuardia Airport,436543
Queens,JFK Airport,343704
Brooklyn,Crown Heights North,262123
Manhattan,Times Sq/Theatre District,234568
Brooklyn,Bushwick South,228560
Manhattan,Midtown Center,224980
Manhattan,East Village,222460
Brooklyn,East New York,221174
Manhattan,TriBeCa/Civic Center,212722
Brooklyn,Williamsburg (North Side),204612


### 7.7 Pickups by borough

In [18]:
%%sql
SELECT
    z.Borough AS borough,
    COUNT(*) AS pickups,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct
FROM trips_clean t
JOIN read_csv_auto('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv') z
  ON t.PULocationID = z.LocationID
GROUP BY z.Borough
ORDER BY pickups DESC;

Running query in 'duckdb'

borough,pickups,share_pct
Manhattan,7120334,35.85
Brooklyn,5432250,27.35
Queens,4451657,22.41
Bronx,2541073,12.79
Staten Island,316459,1.59
N/A,912,0.0


## 8. Notes

* **Plotting SQL results:** capture a result into pandas with `result = %sql SELECT ...` then `result.DataFrame().plot(...)`, or use the `%%sql result <<` syntax.
* **Scaling to multiple months (Part 2 preview):** `read_parquet` accepts a list or glob, e.g. `read_parquet(['...2025-04.parquet', '...2025-05.parquet', '...2025-06.parquet'])`, so the same SQL scales to the full processing set without pandas memory limits.
* **Performance:** every query re-reads from the remote file (with projection pushdown). For repeated analysis, materialise once with `CREATE TABLE trips_local AS SELECT * FROM trips;` and query the local table instead.

## 9. Example: all 2025 months in one query

Two ways to read many monthly files at once:

* **Local files — real glob.** If the files were downloaded (e.g. into `data/`), DuckDB expands the wildcard itself:
  ```sql
  SELECT COUNT(*) FROM read_parquet('../data/fhvhv_tripdata_2025-*.parquet');
  ```
* **Remote HTTPS — explicit URL list.** A plain web server cannot be "listed", so globs do **not** work over `https://`. Instead, build the list of monthly URLs in Python and pass it to `read_parquet`, which accepts a list of files.

The query below unions all 12 months of 2025 and aggregates per month. Note: this scans ~a year of data over the network, so it is much slower than the single-month queries above (DuckDB still only downloads the columns used).


In [23]:
URLS = [
    f"https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_{y}-{m:02d}.parquet"
    for y in range(2019, 2027)
    for m in range(1, 13)
    if not (y == 2019 and m == 1)      # dataset starts 2019-02
    and not (y == 2026 and m > 6)      # adjust to latest published month
]
URLS


['https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-02.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-03.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-04.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-05.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-06.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-07.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-08.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-09.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-10.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-11.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-12.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2020-01.parquet',
 'ht

In [24]:
%%sql
-- {{URLS}} renders the Python list as a DuckDB list of file URLs
SELECT
    EXTRACT(year FROM pickup_datetime)  AS year,
    EXTRACT(month FROM pickup_datetime) AS month,
    COUNT(*) AS trips,
    ROUND(AVG(trip_miles), 2) AS avg_miles,
    ROUND(AVG(base_passenger_fare), 2) AS avg_fare
FROM read_parquet({{URLS}})
GROUP BY year, month
ORDER BY year, month;


Running query in 'duckdb'

HTTPException: HTTP Error: HTTP GET error on 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2023-12.parquet' (HTTP 403 Forbidden)